# Task 1 - data pipeline

Builds the unified 3-class YOLO dataset: **0 cone, 1 barrier, 2 stop_sign**.

| Step | Module | Output |
| --- | --- | --- |
| nuScenes -> YOLO | `data.nuscenes_to_yolo` | labels + a sidecar json per image with `K`, camera height, pitch and ground-truth depth `Z` |
| COCO stop signs | `data.coco_stopsign` | the stop-sign subset of COCO 2017, remapped to class 2 |
| Pseudo-labels | `data.pseudo_label` | stop signs that nuScenes never annotates, so the model is not taught they are background |
| Merge + checks | `data.merge`, `data.sanity_check` | `configs/data.yaml`, instance counts, 20 drawn images |

## Kaggle setup

1. **Settings -> Accelerator: GPU** (only the pseudo-label step needs it).
2. **Settings -> Internet: On** (for `pip install` and, if COCO is not attached, the stop-sign images).
3. **Add Input**: a *nuScenes v1.0-mini* dataset. Any mirror works - the folder is found by its layout (`v1.0-mini/` + `samples/`), not by slug.
4. **Add Input (optional but faster)**: a *COCO 2017* dataset. If it is attached the stop-sign images are symlinked instead of downloaded.

In [ ]:
# Get the repo and the dependencies.
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/heller007/eric-robotics-assessment.git"
ON_KAGGLE = Path("/kaggle/working").exists()

if ON_KAGGLE:
    REPO = Path("/kaggle/working/eric-robotics-assessment")
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
    !pip install -q nuscenes-devkit ultralytics
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("repo:", REPO)

In [ ]:
# What the session can see. Stop here if the nuScenes root is not found.
from common.env import describe, find_coco_root, find_nuscenes_root, writable_root

print(describe())
NUSCENES = find_nuscenes_root()
COCO = find_coco_root()
OUT = writable_root() / "datasets/yolo_3class"
print(f"\nnuScenes : {NUSCENES}")
print(f"COCO     : {COCO or 'not attached -> stop-sign images download over HTTP'}")
print(f"output   : {OUT}")

In [ ]:
# 1. nuScenes -> YOLO. 404 samples x 6 cameras, split by scene (mini_train / mini_val).
!python -m data.nuscenes_to_yolo --nuscenes-root "{NUSCENES}" --out "{OUT}"

In [ ]:
# 2. COCO stop signs -> class 2, into the same tree.
coco_arg = f'--coco-root "{COCO}"' if COCO else ""
!python -m data.coco_stopsign {coco_arg} --out "{OUT}"

In [ ]:
# 3. Pseudo-label the stop signs nuScenes leaves unannotated.
#    Re-running is safe: previous class-2 lines on nuScenes frames are stripped first.
!python -m data.pseudo_label --out "{OUT}" --weights yolo11n.pt --conf 0.5 --imgsz 960 --device 0

In [ ]:
# Spot-check the pseudo-labels before trusting them.
from IPython.display import Image, display

sheet = REPO / "results/pseudo_label_review.jpg"
display(Image(str(sheet))) if sheet.exists() else print("no stop signs were found on nuScenes frames")

In [ ]:
# 4. Validate the merged set and write configs/data.yaml.
!python -m data.merge --out "{OUT}"
!cat configs/data.yaml

In [ ]:
# 5. The checks that must pass before training: counts, per-camera K, positive Z,
#    and 20 drawn images.
!python -m data.sanity_check --out "{OUT}" --draw 20 --dest results/sanity

In [ ]:
# Eyeball the drawn boxes. Labels carry the ground-truth depth where nuScenes has one.
import itertools

for p in itertools.islice(sorted((REPO / "results/sanity/train").glob("*.jpg")), 6):
    display(Image(str(p), width=900))

## Done

Before moving on, confirm:

- both splits report non-zero cone, barrier **and** stop_sign counts;
- the drawn boxes sit on the objects (a normalisation bug shows up immediately here);
- the contact sheet really contains stop signs;
- `sanity_check` printed *all sanity checks passed*.

**Keep this notebook's output**: *Save Version -> Save & Run All*, then add its output as a dataset to the Task 2 notebook, or simply re-run this notebook's cells there.

Next: `task2_detection.ipynb` - B0 zero-shot baseline, then B1 / B1b / B2 fine-tuning.